# Hoist_Collection_Invoicing (cloud migration)

Cloud re-implementation of on-prem `PRC.operation.Hoist_Collection_Invoicing`.

**Grain:** one row per `(ExposureKey, PayDate)` -- a matched collection made inside the assignment window, for the selected SPV portfolios, during the **previous calendar month** relative to `business_date` (empty => previous month relative to today).

**Sources (all pinned, no MAX(BusinessDate) guessing):**
- `derivativeactive.dcollection` -- matched collection amount, portfolio (snapshot = the collection run date).
- `reference.ref_calendar` -- `PreviousBusinessDay` of each `ActualDate`.
- `derivative.dexposure` @ `PreviousBusinessDay` -- canonical `ExposureKey`, CDI, subproduct, BU, buckets, due amount, slave flag.
- `primary.assignment` @ `PreviousBusinessDay`, `SourceSystem IN ('qc','qca')` -- company, assignment/recall dates, bucket, outstanding, `ProcessType` (= on-prem `packet`).
- `derivativeactive.dsettlementexposure` -- settlement debt.

**Join bridge (mirrors `gr.cepal.dna.de.cud/DCollection/CollectionAssignmentHoist.py`):**
`dcollection` -> `ref_calendar` (PreviousBusinessDay) -> `dexposure` @ PBD (ExposureKey both prefixed & bare) -> `primary.assignment` @ PBD (ExposureKey both forms + PortfolioName).
A direct `dcollection.ExposureKey = assignment.ExposureKey` join returns **0 rows** for the SPV portfolios -- the bridge through dexposure is mandatory.

**Key-format note (verified):** QC (pre 2026-06-05) stores `assignment.ExposureKey` **with** the 3-char prefix; QCA (post-decommission) stores it **bare**. The bridge matches on both forms so it works for either era. From Sep-2026 onward all assignment data is QCA.

This notebook **does not write any table** -- it ends by materialising `output_df` (and a `display`) formatted to match the on-prem output exactly (datetime strings, NULL literals, decimal scale, 24-column order) for downstream IT processing.

### On-prem -> cloud column mapping (verified against the SSIS XLSX output)

| On-prem output column | On-prem derivation | Cloud source |
|---|---|---|
| `LoanId` | `Asset.AssetId` | `dcollection.ExposureKey` (prefixed) |
| `CustomerCIF` | `Party.CDI` (Main Beneficiary) | `dexposure.CDI` (fallback `dcollection.PrimaryOwnerCDI`) |
| `AccountNumber` | `Asset.AssetKey` | `dcollection.ExposureKey` (prefixed) |
| `Subproduct` | `Product.SubProductDesc` | `dexposure.SubProductDescription` |
| `AssignmentCompany` | `case_external_assignment.EXTC_NAME` | `assignment.ExternalCompanyFullName` (QCA: `qca_dim_external_company.EXTC_NAME`) |
| `AssignmentDate` | `PACK_ASSIGN_DATE` | `assignment.PacketAssignmentDate` |
| `PayDate` | `Collections.ActualDate` | `dcollection.ActualDate` |
| `PayAmount` | `SUM(AmountEUR) OVER (ActualDate, LoanId)` | `SUM(dcollection.Amount) OVER (ExposureKey, ActualDate)` (computed BEFORE the assignment join, like on-prem `#collections`) |
| `BU` | `Classification.Description` of `LoanMetrics.AssetSegmentId` | `dexposure.AssetSegment` |
| `AssignmentDenounceStatus` | `getCLSDesc(Loan.DenouncedStatusId)` @ AssignmentDate | NULL (on-prem emits NULL in #output; populated by a later point-in-time UPDATE not reconstructable from active snapshot) |
| `PayDateBucketLoan` | `Loan.bucketid` @ PreviousBusinessDay | `dexposure.BucketCode` @ PBD |
| `PayDateBucketSettlement` | `Loan.SettlementBucketId` @ PreviousBusinessDay | `dexposure.SettlementBucketCode` @ PBD |
| `AssignmentBucket` | `case_external_assignment.bucket` | `assignment.AssignmentBucketCode` |
| `AssignmentBucketSettlement` | `Loan.SettlementBucketId` @ AssignmentDate | NULL -- point-in-time at AssignmentDate not reconstructable |
| `DateBackDate` | `NULLIF(PACC_DATEBACK,'9999-12-31')` | `assignment.AssignmentRecallDate` |
| `AssignmentBackBucketLoan` | `Loan.bucketid` @ DateBackDate | NULL -- point-in-time not reconstructable |
| `AssignmentBackBucketSettlement` | `Loan.SettlementBucketId` @ DateBackDate | NULL -- point-in-time not reconstructable |
| `SPV` | `Portfolio.PortfolioName` | `dcollection.PortfolioName` |
| `DebtAmountArchLCYLoan` | `LoanFinancials.Delinquent` @ PreviousBusinessDay | `dexposure.DueAmount` @ PBD |
| `DebtAmountArchLCYSettlement` | `SettlementBucketDaily.DueAmount` @ PreviousBusinessDay | `dsettlementexposure.DebtAmount` (active) |
| `AssignmentOutstandingBalance` | `PACC_OUTSTANDING_AMT` | `assignment.OutstandingAmount` |
| `PreviousBusinessDay` | `Calendar.PreviousBusinessDay` | `ref_calendar.PreviousBusinessDay` |
| `isSlaveAccount` | 1 if in `#slave` (AssetRelationships) | `dexposure.SlaveFlag` cast int |
| `Packet_Process_Type` | `case_external_assignment.packet` | `assignment.ProcessType` (VERIFIED exact match: MRC_Judicial, Mercury 3869 Judicial, Offer, Collection, ...) |
| `sheet` | `datename(mm, prev month)` | month name of the pay period |

In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, lit, broadcast
from pyspark.sql.window import Window
from pyspark.storagelevel import StorageLevel

spark = SparkSession.builder.getOrCreate()

# DEBUG gates the diagnostic .count()/display actions. Each such action re-executes
# the full lineage, so keep it False for production runs; set True only to inspect
# intermediate perimeters during validation.
DEBUG = False

env = aws_utils.get_env_prefix()
if not env:
    _catalogs = {r[0] for r in spark.sql("SHOW CATALOGS").collect()}
    for _p in ["p", "qa", "d"]:
        if f"{_p}-cdh" in _catalogs:
            env = _p
            break
    if not env:
        raise ValueError("Could not resolve env prefix (aws_utils returned None and no *-cdh catalog found)")

src = f"`{env}-cdh`"

# ---- Reference date -------------------------------------------------------
# business_date drives which month is reported. On-prem reports the PREVIOUS
# calendar month relative to GETDATE(); here it is the previous calendar month
# relative to business_date (empty => latest dcollection snapshot date).
business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"{src}.derivativeactive.dcollection")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

# ---- dcollection snapshot to read -----------------------------------------
# The on-prem SP reads the collections table at run time. In cloud we read the
# dcollection snapshot whose BusinessDate = run_date (the day the SP would run).
# Fall back to the latest available snapshot <= run_date if run_date itself is
# not a loaded business day. Direct filter+max (no distinct/shuffle) on the
# BusinessDate partition column -> metadata-only, single job.
COLLECTION_BD = (
    spark.table(f"{src}.derivativeactive.dcollection")
    .filter(col("BusinessDate") <= run_date_lit)
    .agg(F.max("BusinessDate")).collect()[0][0]
)

# ---- Pay period: previous calendar month ----------------------------------
min_pay_date = F.trunc(F.add_months(run_date_lit, -1), "month")
max_pay_date = F.last_day(F.add_months(run_date_lit, -1))

_period = spark.range(1).select(min_pay_date.alias("min_pay"), max_pay_date.alias("max_pay")).collect()[0]
min_pay_str, max_pay_str = str(_period["min_pay"]), str(_period["max_pay"])

# ---- SPV portfolio perimeter ----------------------------------------------
# On-prem @portfolio = DWH PortfolioId 13,41,42,52,53 (Mercury, Orbit, Light,
# Cell, Pearl Portfolio A). Resolve to PortfolioName (the only key consistent
# across dcollection / primary.portfolio) and filter dcollection on the NAME.
ONPREM_PORTFOLIO_IDS = [13, 41, 42, 52, 53]

_pf = spark.table(f"{src}.primary.portfolio")
_pf_bd = _pf.agg(F.max("BusinessDate")).collect()[0][0]
portfolio_map = (
    _pf.filter(col("BusinessDate") == lit(_pf_bd).cast("date"))
    .filter(col("PortfolioId").isin(ONPREM_PORTFOLIO_IDS))
    .select(col("PortfolioId").alias("OnPremPortfolioId"), col("PortfolioName"))
    .dropDuplicates(["PortfolioName"])
)
_pm = portfolio_map.collect()
PERIMETER_NAMES = sorted({r["PortfolioName"] for r in _pm if r["PortfolioName"] is not None})

print(f"env={env}  run_date={run_date}")
print(f"dcollection snapshot (COLLECTION_BD) = {COLLECTION_BD}")
print(f"pay period: {min_pay_str} .. {max_pay_str}")
print(f"on-prem PortfolioIds: {ONPREM_PORTFOLIO_IDS}")
for r in sorted(_pm, key=lambda x: x["OnPremPortfolioId"]):
    print(f"  {r['OnPremPortfolioId']} -> {r['PortfolioName']}")
print(f"perimeter PortfolioNames: {PERIMETER_NAMES}")

In [0]:
# ==================== #collections ====================
# On-prem #collections: dwh.trn.Collections, ActualDate in [MinPayDate, MaxPayDate],
#   FinalMatchedFlag=1, PortfolioId in @portfolio, LoanId not null,
#   PayAmount = SUM(AmountEUR) OVER (ActualDate, LoanId),
#   left join Calendar to get PreviousBusinessDay.
# PayAmount is computed HERE (collection grain) BEFORE any assignment join so the
# assignment fan-out cannot double it.

calendar = (
    spark.table(f"{src}.reference.ref_calendar")
    .select(
        col("CalendarDate").cast("date").alias("cal_date"),
        col("PreviousBusinessDay").cast("date").alias("PreviousBusinessDay"),
    )
    .filter(col("cal_date").isNotNull())
    .dropDuplicates(["cal_date"])
)

_coll = (
    spark.table(f"{src}.derivativeactive.dcollection")
    .filter(col("BusinessDate") == lit(COLLECTION_BD).cast("date"))
    .filter(
        (col("ActualDate") >= min_pay_date)
        & (col("ActualDate") <= max_pay_date)
        & (F.lower(col("FinalReason")) == "matched")
        & (col("PortfolioName").isin(PERIMETER_NAMES))
        & col("ExposureKey").isNotNull()
    )
)

# PayAmount = SUM(Amount) over (ExposureKey, ActualDate) == on-prem SUM over (ActualDate, LoanId)
w_amt = Window.partitionBy("ExposureKey", "ActualDate")

collections_agg = (
    _coll
    .withColumn("PayAmount", F.sum(col("Amount")).over(w_amt))
    .select(
        col("ExposureKey"),
        col("PortfolioName"),
        col("ActualDate").cast("date").alias("PayDate"),
        col("PayAmount").cast("decimal(19,4)").alias("PayAmount"),
        col("PrimaryOwnerCDI"),
        col("AllocationExternalCompany"),
    )
    .distinct()
)

# attach PreviousBusinessDay for the PBD-pinned joins that follow.
# calendar is a tiny dimension -> broadcast to skip a shuffle.
# Normalize the collection key to its BARE form ONCE here (collection keys are
# always prefixed) so every downstream join is a single-key equi-join (no OR).
collections_cal = (
    collections_agg.join(broadcast(calendar), collections_agg["PayDate"] == calendar["cal_date"], "left")
    .drop("cal_date")
    .withColumn("coll_bare", F.expr("substring(ExposureKey, 4, 100)"))
)

# collections_cal is the hot DataFrame: it feeds the PBD list, the dexposure join
# and (transitively) every downstream stage. Persist it so the lineage above
# (scan + window + distinct + calendar join) runs once, not once per consumer.
collections_cal = collections_cal.persist(StorageLevel.MEMORY_AND_DISK)

if DEBUG:
    print(f"matched collections in period (ExposureKey x PayDate): {collections_cal.count():,}")

In [0]:
# ==================== dexposure @ PreviousBusinessDay ====================
# On-prem #loaninfo (Subproduct, BU, CDI, isSlave) + buckets/debt @ PreviousBusinessDay.
# Cloud dexposure unifies all of this. Read it at the PreviousBusinessDay of each
# collection (the same snapshot the CUD CollectionAssignmentHoist bridge uses), so
# the ExposureKey resolution and the bucket/debt values are as-of PBD, matching on-prem.
#
# dexposure gives the CANONICAL ExposureKey used to reach primary.assignment.
# dexposure keys are always PREFIXED -> normalize to BARE (dx_bare) so the join to
# the collection's bare key is a single-key equi-join (no OR -> hash-joinable,
# enables bloom-filter pushdown). Same portfolio => same prefix, so matching on the
# bare key + PortfolioName is equivalent to the old prefixed-OR-bare predicate.
_pbd_values = [r[0] for r in collections_cal.select("PreviousBusinessDay").distinct().collect() if r[0] is not None]

dexposure = (
    spark.table(f"{src}.derivative.dexposure")
    .filter(col("BusinessDate").isin(_pbd_values))
    .select(
        F.expr("substring(ExposureKey, 4, 100)").alias("dx_bare"),
        col("ExposureKey").alias("dx_ExposureKey"),
        col("BusinessDate").alias("dx_BusinessDate"),
        col("PortfolioName").alias("dx_PortfolioName"),
        col("CDI").alias("dx_CDI"),
        col("SubProductDescription").alias("dx_Subproduct"),
        col("AssetSegment").alias("dx_BU"),
        col("SlaveFlag").alias("dx_SlaveFlag"),
        col("BucketCode").alias("dx_BucketCode"),
        col("SettlementBucketCode").alias("dx_SettlementBucketCode"),
        col("DueAmount").alias("dx_DueAmount"),
    )
    .dropDuplicates(["dx_ExposureKey", "dx_BusinessDate"])
)

# bridge: single-key equi-join on (PBD, PortfolioName, bare key). No OR.
enriched = collections_cal.join(
    dexposure,
    (collections_cal["PreviousBusinessDay"] == dexposure["dx_BusinessDate"])
    & (collections_cal["PortfolioName"] == dexposure["dx_PortfolioName"])
    & (collections_cal["coll_bare"] == dexposure["dx_bare"]),
    "left",
)

if DEBUG:
    print(f"after dexposure bridge: {enriched.count():,}")
    print(f"  with dexposure match: {enriched.filter(col('dx_ExposureKey').isNotNull()).count():,}")

In [0]:
# ==================== Settlement debt (DebtAmountArchLCYSettlement) ====================
# On-prem: Analytic.bi.SettlementBucketDaily.DueAmount @ PreviousBusinessDay.
# Cloud:   dsettlementexposure.DebtAmount for the active settlement per ExposureKey.
dse_bd = (
    spark.table(f"{src}.derivativeactive.dsettlementexposure")
    .agg(F.max("BusinessDate")).collect()[0][0]
)

settlement = (
    spark.table(f"{src}.derivativeactive.dsettlementexposure")
    .filter(col("BusinessDate") == lit(dse_bd).cast("date"))
    .filter(col("IsActiveFlag") == 1)
    .groupBy(col("ExposureKey").alias("se_ExposureKey"))
    .agg(F.sum(col("DebtAmount")).cast("decimal(19,2)").alias("se_DebtAmountSettlement"))
)

# join on the canonical dexposure key (fallback to the collection key when no dexposure match).
# settlement is a small pre-aggregated dim -> broadcast to skip a shuffle.
enriched = enriched.withColumn(
    "canon_key", F.coalesce(col("dx_ExposureKey"), col("ExposureKey"))
)
enriched = enriched.join(
    broadcast(settlement), enriched["canon_key"] == settlement["se_ExposureKey"], "left"
)

In [0]:
# ==================== Assignment (primary.assignment @ PreviousBusinessDay) ====================
# On-prem #assignment = DWH_STG.qc.case_external_assignment joined to #collections on
#   AssetId and the window [PACK_ASSIGN_DATE .. isnull(PACC_DATEBACK,'9999-12-31')].
# The commented line `--where a.packet='collection' ASK-4317` proves the ProcessType
# restriction was DELIBERATELY REMOVED -> we do NOT filter on ProcessType or AssignmentType.
#
# Cloud bridge (from CUD CollectionAssignmentHoist.py):
#   assignment read @ a.BusinessDate = collection.PreviousBusinessDay,
#   a.PortfolioName = collection.PortfolioName,
#   a.SourceSystem IN ('qc','qca'),
#   key match: dexposure.ExposureKey (prefixed) = a.ExposureKey  OR  bare = a.ExposureKey.
# QC stores assignment.ExposureKey WITH prefix, QCA stores it BARE -> match both forms.
#
# An exposure can have several assignment rows on one snapshot. Keep ONE per
# (ExposureKey, BusinessDate) by assignmentprocess priority (collection > offer >
# officer > other) -- identical to the CUD dedup -- so the assignment join cannot
# fan out the collection grain.
#
# Field mapping (on-prem -> primary.assignment):
#   EXTC_NAME            -> ExternalCompanyFullName
#   PACK_ASSIGN_DATE     -> PacketAssignmentDate
#   PACC_DATEBACK        -> AssignmentRecallDate
#   bucket               -> AssignmentBucketCode
#   PACC_OUTSTANDING_AMT -> OutstandingAmount
#   packet               -> ProcessType   (VERIFIED exact match to XLSX Packet_Process_Type)

# CRITICAL: restrict to AssignmentType = 'Collection Assignment'. These are the
# cloud equivalent of on-prem case_external_assignment and are the ONLY rows that
# carry AssignmentBucketCode + OutstandingAmount (verified: 100% populated on CA
# rows, 0% on 'Channel Assignment'). The assignment-window match also resolves
# exclusively to CA rows, so this filter does NOT shrink the perimeter -- it only
# ensures bucket/outstanding/dates come from the correct record. (This is distinct
# from the obsolete ProcessType='Collection' filter removed per ASK-4317.)
assignment_all = (
    spark.table(f"{src}.primary.assignment")
    .filter(col("BusinessDate").isin(_pbd_values))
    .filter(col("SourceSystem").isin("qc", "qca"))
    .filter(col("AssignmentType") == "Collection Assignment")
    .filter(col("ExposureKey").isNotNull())
)

# One Collection-Assignment row per (ExposureKey, BusinessDate, PortfolioName):
# keep the latest by PacketAssignmentDate (the current assignment window).
w_assign = Window.partitionBy("ExposureKey", "BusinessDate", "PortfolioName").orderBy(
    col("PacketAssignmentDate").desc_nulls_last(),
)

assignment = (
    assignment_all
    .withColumn("_rn", F.row_number().over(w_assign))
    .filter(col("_rn") == 1)
    .select(
        col("ExposureKey").alias("as_ExposureKey"),
        col("BusinessDate").alias("as_BusinessDate"),
        col("PortfolioName").alias("as_PortfolioName"),
        col("ExternalCompanyFullName").alias("as_AssignmentCompany"),
        col("PacketAssignmentDate").cast("date").alias("as_AssignmentDate"),
        col("AssignmentRecallDate").cast("date").alias("as_DateBackDate"),
        col("AssignmentBucketCode").alias("as_AssignmentBucket"),
        col("OutstandingAmount").alias("as_OutstandingAmount"),
        col("ProcessType").alias("as_ProcessType"),
    )
    # Normalize the assignment key to BARE once (VERIFIED against prod):
    #   QCA stores it BARE (starts with a digit) -> keep as-is.
    #   QC  stored it PREFIXED (3 uppercase letters) -> strip the 3-char prefix.
    # The RLIKE('^[A-Z]{3}') guard distinguishes the two. Both sides then live in
    # the same bare key space, so the join is a single-key equi-join (no OR).
    .withColumn(
        "as_bare",
        F.when(col("as_ExposureKey").rlike("^[A-Z]{3}"), F.expr("substring(as_ExposureKey, 4, 100)"))
         .otherwise(col("as_ExposureKey")),
    )
)

# collection-side canonical bare key (canon_key is prefixed -> strip to bare)
enriched = enriched.withColumn("canon_bare", F.expr("substring(canon_key, 4, 100)"))

# bridge: single-key equi-join on (PBD, PortfolioName, bare key). No OR.
# assignment is deduped to ~1 row per exposure (small) -> broadcast.
enriched = enriched.join(
    broadcast(assignment),
    (enriched["PreviousBusinessDay"] == assignment["as_BusinessDate"])
    & (enriched["PortfolioName"] == assignment["as_PortfolioName"])
    & (enriched["canon_bare"] == assignment["as_bare"]),
    "left",
)

if DEBUG:
    print(f"after assignment bridge: {enriched.count():,}")
    print(f"  with assignment match: {enriched.filter(col('as_ExposureKey').isNotNull()).count():,}")

In [0]:
# ==================== Final output ====================
# On-prem #output INNER-joins #collections to #assignment on the window
#   c.ActualDate >= AssignmentDate AND c.ActualDate <= isnull(DateBack,'9999-12-31').
# STRICT_ASSIGNMENT_WINDOW=True reproduces that INNER join (drop collections whose
# PayDate is outside the assignment window / have no assignment). Toggle to False
# during validation to inspect the pre-window perimeter.
STRICT_ASSIGNMENT_WINDOW = True

sheet_name = spark.range(1).select(
    F.date_format(F.add_months(run_date_lit, -1), "MMMM").alias("m")
).collect()[0]["m"]

in_window = (
    col("as_AssignmentDate").isNotNull()
    & (col("PayDate") >= col("as_AssignmentDate"))
    & (col("PayDate") <= F.coalesce(col("as_DateBackDate"), lit("9999-12-31").cast("date")))
)

enriched_win = enriched.filter(in_window) if STRICT_ASSIGNMENT_WINDOW else enriched

# ---- Build raw typed columns (pre-format) ----
raw = enriched_win.select(
    col("ExposureKey").alias("LoanId"),
    F.coalesce(col("dx_CDI"), col("PrimaryOwnerCDI")).alias("CustomerCIF"),
    col("ExposureKey").alias("AccountNumber"),
    col("dx_Subproduct").alias("Subproduct"),
    F.coalesce(col("as_AssignmentCompany"), col("AllocationExternalCompany")).alias("AssignmentCompany"),
    col("as_AssignmentDate").alias("AssignmentDate"),
    col("PayDate").alias("PayDate"),
    col("PayAmount").cast("decimal(19,4)").alias("PayAmount"),
    col("dx_BU").alias("BU"),
    lit(None).cast("string").alias("AssignmentDenounceStatus"),
    col("dx_BucketCode").cast("string").alias("PayDateBucketLoan"),
    col("dx_SettlementBucketCode").cast("string").alias("PayDateBucketSettlement"),
    col("as_AssignmentBucket").cast("int").alias("AssignmentBucket"),
    lit(None).cast("int").alias("AssignmentBucketSettlement"),
    col("as_DateBackDate").alias("DateBackDate"),
    lit(None).cast("int").alias("AssignmentBackBucketLoan"),
    lit(None).cast("int").alias("AssignmentBackBucketSettlement"),
    col("PortfolioName").alias("SPV"),
    col("dx_DueAmount").cast("decimal(19,2)").alias("DebtAmountArchLCYLoan"),
    col("se_DebtAmountSettlement").cast("decimal(19,2)").alias("DebtAmountArchLCYSettlement"),
    col("as_OutstandingAmount").cast("decimal(19,4)").alias("AssignmentOutstandingBalance"),
    col("PreviousBusinessDay").alias("PreviousBusinessDay"),
    F.coalesce(col("dx_SlaveFlag").cast("int"), lit(0)).alias("isSlaveAccount"),
    col("as_ProcessType").alias("Packet_Process_Type"),
).distinct()

# ---- Exact on-prem string formatting for IT delivery ----
# Dates -> 'yyyy-MM-dd HH:mm:ss.SSS'; NULLs -> literal 'NULL'; decimals -> fixed scale.
def _dt(c):
    return F.coalesce(F.date_format(col(c).cast("timestamp"), "yyyy-MM-dd HH:mm:ss.SSS"), lit("NULL"))

def _nullstr(c):
    return F.coalesce(col(c).cast("string"), lit("NULL"))

def _dec(c, scale):
    return F.coalesce(F.format_number(col(c).cast("decimal(19,4)"), scale), lit("NULL"))

output_df = raw.select(
    lit(sheet_name).alias("sheet"),
    _nullstr("LoanId").alias("LoanId"),
    _nullstr("CustomerCIF").alias("CustomerCIF"),
    _nullstr("AccountNumber").alias("AccountNumber"),
    _nullstr("Subproduct").alias("Subproduct"),
    _nullstr("AssignmentCompany").alias("AssignmentCompany"),
    _dt("AssignmentDate").alias("AssignmentDate"),
    _dt("PayDate").alias("PayDate"),
    _dec("PayAmount", 2).alias("PayAmount"),
    _nullstr("BU").alias("BU"),
    _nullstr("AssignmentDenounceStatus").alias("AssignmentDenounceStatus"),
    _nullstr("PayDateBucketLoan").alias("PayDateBucketLoan"),
    _nullstr("PayDateBucketSettlement").alias("PayDateBucketSettlement"),
    _nullstr("AssignmentBucket").alias("AssignmentBucket"),
    _nullstr("AssignmentBucketSettlement").alias("AssignmentBucketSettlement"),
    _dt("DateBackDate").alias("DateBackDate"),
    _nullstr("AssignmentBackBucketLoan").alias("AssignmentBackBucketLoan"),
    _nullstr("AssignmentBackBucketSettlement").alias("AssignmentBackBucketSettlement"),
    _nullstr("SPV").alias("SPV"),
    _dec("DebtAmountArchLCYLoan", 2).alias("DebtAmountArchLCYLoan"),
    _dec("DebtAmountArchLCYSettlement", 2).alias("DebtAmountArchLCYSettlement"),
    _dec("AssignmentOutstandingBalance", 4).alias("AssignmentOutstandingBalance"),
    _dt("PreviousBusinessDay").alias("PreviousBusinessDay"),
    _nullstr("isSlaveAccount").alias("isSlaveAccount"),
    _nullstr("Packet_Process_Type").alias("Packet_Process_Type"),
)

if DEBUG:
    print(f"output_df rows: {output_df.count():,}")

display(output_df)

# ---- Export the materialised output to a UC Volume file -------------------
# Read this notebook's own config (output_file_prefix + report_output) and write
# output_df to the delivery Volume via the shared export_data helper. Data logic
# above is unchanged; this only materialises the result to a file.
import os, json
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_report_root = "/Workspace" + _nb_path.rsplit("/", 1)[0]   # .../HoistInvoicing
_cfg_path = f"{_report_root}/config/live/hoist_collection_invoicing.json"
with open(_cfg_path) as _f:
    _cfg = json.load(_f)["report_args"]
output_file_prefix = _cfg["output_file_prefix"]   # "hoistinvoicing"
report_output = _cfg["report_output"]             # "internalreports/hoistinvoicing/"

output_dir = f"/Volumes/{env}-cdh-delivery/s3_external/reports/{report_output}{run_date.replace('-', '')}/"
os.makedirs(output_dir, exist_ok=True)

_repo_root = _report_root.rsplit("/HoistInvoicing", 1)[0]
sys.path.append(f"{_repo_root}/01-utils")
from export_utils import export_data
export_data(output_df, output_file_prefix, run_date, output_dir, output_file_prefix)

# release the cached hot DataFrame now the pipeline has materialised
collections_cal.unpersist()

## Known differences vs on-prem

The assignment-record fields (company, assignment/recall dates, bucket, outstanding, ProcessType) are recovered from `primary.assignment` via the `dcollection -> dexposure(@PBD) -> assignment(@PBD)` bridge. From Sep-2026 onward all assignment data is QCA (`ExternalCompanyFullName` <- `qca_dim_external_company.EXTC_NAME`). The remaining gaps are the **point-in-time Loan/Settlement bucket** values that on-prem reconstructs by range-joining `DWH.ast.Loan` on `EffectiveDate <= date < EndDate`, because the cloud tables are active snapshots.

| Field | On-prem | Cloud behaviour | Status |
|---|---|---|---|
| `Packet_Process_Type` | `case_external_assignment.packet` | `primary.assignment.ProcessType` | Verified exact match |
| `AssignmentCompany` | `EXTC_NAME` | `assignment.ExternalCompanyFullName` (QCA) / fallback `dcollection.AllocationExternalCompany` | QC judicial/3869 packets had NULL company in cloud; QCA populates it |
| `AssignmentDate` / `DateBackDate` / `AssignmentBucket` / `AssignmentOutstandingBalance` | case_external_assignment | `assignment.PacketAssignmentDate` / `AssignmentRecallDate` / `AssignmentBucketCode` / `OutstandingAmount` | Recovered |
| `AssignmentDenounceStatus` | `getCLSDesc(Loan.DenouncedStatusId)` @ AssignmentDate | NULL | on-prem #output also seeds NULL then UPDATEs point-in-time; not reconstructable from active snapshot |
| `AssignmentBucketSettlement` / `AssignmentBackBucketLoan` / `AssignmentBackBucketSettlement` | Loan buckets @ AssignmentDate / DateBackDate | NULL | point-in-time at a past date not reconstructable |
| `PayDateBucketLoan` / `PayDateBucketSettlement` / `DebtAmountArchLCYLoan` | Loan/LoanFinancials @ PreviousBusinessDay | dexposure @ PBD | Recovered (dexposure read at PBD) |
| `DebtAmountArchLCYSettlement` | SettlementBucketDaily.DueAmount @ PBD | dsettlementexposure.DebtAmount (active snapshot) | Snapshot approximation |
| `LoanId` / `AccountNumber` | DWH numeric AssetId / AssetKey | ExposureKey (prefixed) | By design (cloud has no DWH numeric id) |

**Validation:** run per `platform-migration-checklist` (row count, portfolio distribution, financial totals) against the on-prem SP output for the same month before delivery. Toggle `STRICT_ASSIGNMENT_WINDOW=False` to inspect the pre-window perimeter if counts diverge.